# Descarga de Datos PT1

Una vez creamos el diccionario maestro. El que almacena toda la información del DGSIAP que utilizaremos vamos a construir la herramienta que nos permitira descargar los archivos de forma programatica.

In [1]:
# Hacemos la conexion con la pagina de DGSIAP. Además de seleccionar la categoria de interés
from scripts import dgsiap
from scripts import downassist

raw_data = dgsiap.dict_master()


Bienvenido al script de descarga de datos abiertos del DGSIAP.
Conexion Establecida
Los datos historicos se presentan en el espacio denomiado: Información por nivel
Los datos se encuentran divididos en 3 bloques:

1. Municipal
2. Nacional
3. No seguimiento
{'metadata': {'title': 'Metadatos Principales', 'state': 'Actualizado', 'lastdate': '29 julio 2025', 'range': 'Cobertura 2020 - 2021', 'size': 'Nivel Nacional'}, 'info': {'title': 'Diccionario de datos', 'description': 'Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.', 'download': '/index.php?view=3DA7A2BB-7AEC2838-9D81CAA7-4EAF9CAD'}}
{'metadata': {'title': 'Metadatos Principales', 'state': 'Actualizado', 'lastdate': '12 agosto 2026', 'range': 'Cobertura 2003 - 2025', 'size': 'Nivel Municipal'}, 'info': {'title': 'Diccionario de datos', 'description': 'Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases

## Herramienta Auxiliar

In [2]:
print(raw_data['No seguimiento'])
print(raw_data['Municipal'])

{'Metadata': {'metadata': {'title': 'Metadatos Principales', 'state': 'Actualizado', 'lastdate': '29 julio 2025', 'range': 'Cobertura 2020 - 2021', 'size': 'Nivel Nacional'}, 'info': {'title': 'Diccionario de datos', 'description': 'Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.', 'download': '/index.php?view=3DA7A2BB-7AEC2838-9D81CAA7-4EAF9CAD'}}, 'Datasets': [URL('/index.php?view=4877B98E-DC2C7731-96F54D97-B642E9D1&ANIO=2021'), URL('/index.php?view=4877B98E-DC2C7731-96F54D97-B642E9D1&ANIO=2020')]}
{'Metadata': {'metadata': {'title': 'Metadatos Principales', 'state': 'Actualizado', 'lastdate': '12 agosto 2026', 'range': 'Cobertura 2003 - 2025', 'size': 'Nivel Municipal'}, 'info': {'title': 'Diccionario de datos', 'description': 'Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.', 'download': '/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D

## Selector de información

Una vez hayamos obtenido el diccionario maestro. Construiremos el selector de información. Despues de escoger la categoria. Notemos que solo podemos elegir el bloque (Los que esten enumerados arriba) y el año de cierre.

Los bloques podemos verlos arriba. Entonces nos enfocaremos en el año.

In [3]:
def when(**kwargs):
    """
        De momento solo funcionara con el diccionario creado por dgsiap.py
    """
    year_c = kwargs.get("year_column", "ANIO")
    years = {}
    for b, info in raw_data.items():
        history = info['Datasets']
        history_links = [downassist.parse_link(url) for url in history]
        links_params = [l.query['ANIO'] for l in history_links]
        years[b] = links_params
    return years

when()
        

{'Municipal': ['2025',
  '2024',
  '2023',
  '2022',
  '2021',
  '2020',
  '2019',
  '2018',
  '2017',
  '2016',
  '2015',
  '2014',
  '2013',
  '2012',
  '2011',
  '2010',
  '2009',
  '2008',
  '2007',
  '2006',
  '2005',
  '2004',
  '2003'],
 'Nacional': ['2002',
  '2001',
  '2000',
  '1999',
  '1998',
  '1997',
  '1996',
  '1995',
  '1994',
  '1993',
  '1992',
  '1991',
  '1990',
  '1989',
  '1988',
  '1987',
  '1986',
  '1985',
  '1984',
  '1983',
  '1982',
  '1981',
  '1980'],
 'No seguimiento': ['2021', '2020']}

Cabe aclarar que la función `when()` es para exploracion previa. Si por otro lado prefieres, ya conocidos los bloques, preguntar por cualquier periodo y que solo se te devuelva los años disponibles usaremos la siguiente.

In [4]:
def input_parser(value):
    """
        Elementos string, int o list los coniverte a lista
    """
    if not type(value)==list:
        assert type(value)==str or type(value)==int, "Revisa tu entrada"
        value = [value]
    try:
        list_value = [str(v) for v in value]
    except e:
        print(e, "Revisa tu seleccion")
    return list_value


def year_checker(category, period, **kwargs):
    """
        Si la dejas en blanco considerara todos los años disponibles
    """
    assert category in raw_data.keys(), "La categoria no existe"
    y_c = kwargs.get("year_column", "ANIO")
    unique = kwargs.get("deep", False)
    years = input_parser(period)
    selection = {y: [] for y in years}
    subset = raw_data[category]['Datasets']
    for y in years:
            for url in subset:
                checker = 1 if str(y)==url.query['ANIO'] else 0
                if checker:
                    selection[y].append(url)
                    if unique:
                        break
    return selection

year_checker(category="Municipal", period=[1800,"2020"])

{'1800': [],
 '2020': [URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2020')]}

Concluyendo nuestro selector de información para descargar los conjuntos de datos

In [5]:
def selector(category, period, **kwargs):
    selection = {}
    category = input_parser(category)
    for c in category:
        assert c in raw_data.keys(), "Revisa tus categorias"
    for cat in category:
        selection[cat] = year_checker(cat, period, **kwargs)
    return selection

A lo anterior debemos agregar el diccionario de datos y obtenemos nuestro diccionario conector. El que usaremos para indicar de forma programatica que cosas vamos a descargar

In [6]:
def down_arq(category, period):
    request = {}
    category = input_parser(category)
    for c in category:
        assert c in raw_data.keys(), "Revisa tus categorias"
        dict_cat = raw_data[c]['Metadata']['info']['download']
        data_cat = selector(category, period)
        req_cat = {"Codebook":dict_cat,
                    "Datasets":data_cat[c]}
        request[c] = req_cat
    return request
down_arq("Municipal", [2019, 2020])['Municipal']

{'Codebook': '/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7',
 'Datasets': {'2019': [URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2019')],
  '2020': [URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2020')]}}

Notemos que con esto tenemos una seleccion precisa de lo que necesitamos. Ya sea por bloque o por año. Pero por nuestra premisa desconocemos los años disponibles, entonces veremos como descargar categorias completas y finalmente todo los datos. 

In [ ]:
def full_cat(category):
    """
        Descargar una categoria completa.
    """
    category = input_parser(category)
    for c in category:
        assert c in raw_data.keys(), "Revisa las categorias"
    response = {}
    for c in category:
        cat_code = raw_data[c]['Metadata']['info']['download']
        cat_data = raw_data[c]['Datasets']
        response[c] = {"Codebook": cat_code, "Data":cat_data}
    return response


def downassist(**kwargs):
    """
        Descarga arbitraria incompleta de información del DGSIAP
    """
    request = kwargs.get("request", "*")
    response = None
    if request == "*":
        response = full_cat(list(raw_data.keys()))
        return response
    assert set(request.keys()) == set(["category", "period"]), "Revisa tu solicitud"
    cats = input_parser(request['category'])
    years = input_parser(request['period'])
    if cats[0]=="*":
        response ={}
        for c in raw_data.keys():
            response[c]=down_arq(c, years)[c]
        return response
    if years[0] == "*":
        response = full_cat(cats)
    else:
        response = down_arq(cats, years)
    return response

Ahora considere el siguiente ejemplo. Vemos como el asistente nos genera

* El nombre del bloque.
* Los enlaces para descargar
    * Codebook: El diccionario de datos
    * Data: Los conjuntos de datos asociados al bloque seleccionado.

In [24]:
test_response = {"category":"*", "period": [2019, "2020"]}
dfile= downassist(request= test_response)
print(dfile)

{'Municipal': {'Codebook': '/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7', 'Datasets': {'2019': [URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2019')], '2020': [URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2020')]}}, 'Nacional': {'Codebook': '/index.php?view=3494D63A-569A6D70-7435040E-4965D1CB', 'Datasets': {'2019': [], '2020': []}}, 'No seguimiento': {'Codebook': '/index.php?view=3DA7A2BB-7AEC2838-9D81CAA7-4EAF9CAD', 'Datasets': {'2019': [], '2020': [URL('/index.php?view=4877B98E-DC2C7731-96F54D97-B642E9D1&ANIO=2020')]}}}


# Conclusiones

Finalmente. Generada la solicitud. Procdemos con la obtención de archivos.

In [40]:
from scripts.downassist import parse_link, get_filename, download


def downbot(dictlink):
    all_links = []
    all_fns = []
    for c in dictlink.keys():
        cat_links=dictlink[c]
        for dt in cat_links.keys():
            set_url = cat_links[dt]
            if not type(set_url) == list:
                link = parse_link(set_url)
                all_links.append(link)
                if c=="No seguimiento":
                    all_fns.append(get_filename(link))
                else:
                    all_fns.append(f"{c[:2]}_{get_filename(link)}")
            else:
                seturl = []
                for lsurl in set_url.values():
                    seturl += lsurl
                lslink = [parse_link(url) for url in seturl]
                all_links += lslink
                all_fns +=[f"{get_filename(link)}" for link in lslink]
            for dl in range(len(all_links)):
                filename = all_fns[dl]
                downlink = all_links[dl]
                print(filename)
                download(downlink, filename)

# downbot(dfile)



In [41]:
agro = downassist()

In [ ]:
print(agro)


def full_data():
    down_data = {}
    agro = downassist()
    for block, seturl in agro.items():
        for dt, url in seturl.items():
            if not type(url)==list:
                link = parse_link(url)
                fn = get_filename(link)
                if block != "No segumiento":
                    fn = f"{block[:2]}_{fn}"
                down_data[fn] = link
            else:
                links = [parse_link(u) for u in url]
                fns = [get_filename(l) for l in links]
                for k in range(len(links)):
                    down_data[fns[k]] = links[k]
        for name, link in down_data.items():
            download(link, name)
full_data()

# 4 minutos descargar todo. 30s verificar todos los archivos del DGSIAP

{'Municipal': {'Codebook': '/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7', 'Data': [URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2025'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2024'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2023'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2022'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2021'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2020'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2019'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2018'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2017'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2016'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2015'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2014'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2013'),